In [1]:
# uv venv --seed --python 3.12.13 .venv
# pip install -r requirements.txt

In [1]:
# !pytest tests/test_smoke.py

In [4]:
# !pytest tests/test_cases.py -qq

In [5]:
from feasibility.models import Client, CreditorRules, Offer
from feasibility.engine import Result


def evaluate_offer(client: Client, offer: Offer, rules: CreditorRules) -> Result:
    """Evaluate a single offer. See ASSIGNMENT.md for the full specification.

    Return a Result with feasible=True and a schedule when the offer fits, or
    feasible=False with additional_funds (minimum lump sum AND minimum monthly
    increment) when it does not.
    """
    raise NotImplementedError("Implement evaluate_offer — see ASSIGNMENT.md")

In [6]:
from feasibility.models import load_case

client, offer, rules = load_case(f"cases/case1_feasible_even")

In [7]:
offer

Offer(creditor='EvenCo', current_balance_cents=100000, original_balance_cents=120000, settlement_pct=0.5, first_payment_date=datetime.date(2026, 1, 31))

In [8]:
rules

CreditorRules(max_terms=6, max_payments=6, min_payment_cents=2500, max_token_pays=6, min_payment_tiers=[], even_pays=True, is_ballooning_allowed=False, max_segments=1, bank_fee_cents=1000, program_fee_pct=0.25)

In [9]:
total_offer = offer.current_balance_cents * offer.settlement_pct
total_offer

50000.0

In [10]:
program_fee = offer.original_balance_cents * rules.program_fee_pct
program_fee

30000.0

In [11]:
50000 // 6, 50000 % 6

(8333, 2)

In [12]:
import math

nums = [5000.1, 5000.49999, 5000.500, 5000.5001, 5000.99]

for num in nums:
    fraction = num - int(num)
    if fraction < 0.5:
        print(f"{num} --> {math.floor(num)}")
    else:
        print(f"{num} --> {math.ceil(num)}")

5000.1 --> 5000
5000.49999 --> 5000
5000.5 --> 5001
5000.5001 --> 5001
5000.99 --> 5001


In [13]:
import math


def round_half_up(x: float | int):
    fraction = x - int(x)
    if fraction < 0.5:
        return math.floor(x)
    else:
        return math.ceil(x)

In [14]:
for num in nums:
    print(f"{num} --> {round_half_up(num)}")

5000.1 --> 5000
5000.49999 --> 5000
5000.5 --> 5001
5000.5001 --> 5001
5000.99 --> 5001


In [15]:
int(5000.5)

5000

In [16]:
total_offer = round_half_up(offer.current_balance_cents * offer.settlement_pct)
total_offer

50000

In [17]:
program_fee = round_half_up(offer.original_balance_cents * rules.program_fee_pct)
program_fee

30000

In [18]:
def _run(case: str):
    client, offer, rules = load_case(f"cases/{case}")
    return evaluate_offer(client, offer, rules)


def test_case1_feasible_even():
    r = _run("case1_feasible_even")
    assert r.feasible is True
    assert r.pay_shape_used == "even"
    assert r.schedule is not None
    # balance must never go negative
    assert all(row.balance_cents >= 0 for row in r.schedule)

In [19]:
# test_case1_feasible_even()

In [20]:
def evaluate_offer(client: Client, offer: Offer, rules: CreditorRules) -> Result:
    pass

In [21]:
k_max = min(rules.max_terms, rules.max_payments)
if rules.even_pays:
    base_amount = total_offer // k_max
    remaining_cents = total_offer % k_max

    creditor_payments = [base_amount] * k_max

    for i in range(remaining_cents):
        creditor_payments[k_max - 1 - i] += 1

creditor_payments

[8333, 8333, 8333, 8333, 8334, 8334]

In [22]:
client.as_of_date <= client.last_draft_date

True

In [23]:
import datetime


def get_last_day_in_month(year, month):
    first_day = datetime.date(year, month, 1)
    iter_day = first_day

    while (iter_day + datetime.timedelta(days=1)).month == month:
        iter_day = iter_day + datetime.timedelta(days=1)

    return iter_day.day


def next_month(d: datetime.date, is_last_day_celing, anchor_day):
    if d.month == 12:
        year, month = d.year + 1, 1
    else:
        year, month = d.year, d.month + 1

    last_day_of_month = get_last_day_in_month(year, month)

    if is_last_day_celing:
        return datetime.date(year, month, last_day_of_month)

    return datetime.date(year, month, min(anchor_day, last_day_of_month))


cadence_dates = []
first_payment_date = offer.first_payment_date
iter_date = first_payment_date
i = 0

is_last_day_ceiling = (
    get_last_day_in_month(first_payment_date.year, first_payment_date.month)
    == first_payment_date.day
)

anchor_day = first_payment_date.day

while i < k_max:
    i += 1
    cadence_dates.append(iter_date)
    iter_date = next_month(iter_date, is_last_day_ceiling, anchor_day)

In [24]:
cadence_dates

[datetime.date(2026, 1, 31),
 datetime.date(2026, 2, 28),
 datetime.date(2026, 3, 31),
 datetime.date(2026, 4, 30),
 datetime.date(2026, 5, 31),
 datetime.date(2026, 6, 30)]

In [25]:
date_to_draft_map = {}

for draft in client.ledger:
    date_to_draft_map[draft.date] = draft

In [26]:
date_to_draft_map

{datetime.date(2026, 1, 1): LedgerEntry(date=datetime.date(2026, 1, 1), amount_cents=20000, type='credit'),
 datetime.date(2026, 2, 1): LedgerEntry(date=datetime.date(2026, 2, 1), amount_cents=20000, type='credit'),
 datetime.date(2026, 3, 1): LedgerEntry(date=datetime.date(2026, 3, 1), amount_cents=20000, type='credit'),
 datetime.date(2026, 4, 1): LedgerEntry(date=datetime.date(2026, 4, 1), amount_cents=20000, type='credit'),
 datetime.date(2026, 5, 1): LedgerEntry(date=datetime.date(2026, 5, 1), amount_cents=20000, type='credit'),
 datetime.date(2026, 6, 1): LedgerEntry(date=datetime.date(2026, 6, 1), amount_cents=20000, type='credit'),
 datetime.date(2026, 7, 1): LedgerEntry(date=datetime.date(2026, 7, 1), amount_cents=20000, type='credit')}

In [27]:
def get_last_day_in_month(year, month):
    first_day = datetime.datetime(year, month, 1)
    iter_day = first_day

    while (iter_day + datetime.timedelta(days=1)).month == month:
        iter_day = iter_day + datetime.timedelta(days=1)

    return iter_day


def next_month(d: datetime.datetime, is_last_day_celing):
    if d.month == 12:
        year, month = d.year + 1, 1
    else:
        year, month = d.year, d.month + 1

    last_day_of_month = get_last_day_in_month(year, month)

    if is_last_day_celing:
        return datetime.datetime(year, month, last_day_of_month)

    return datetime.datetime(year, month, min(d.day, last_day_of_month))

In [28]:
movement_days = cadence_dates + list(date_to_draft_map.keys())

In [29]:
movement_days = sorted(movement_days)

In [30]:
movement_days

[datetime.date(2026, 1, 1),
 datetime.date(2026, 1, 31),
 datetime.date(2026, 2, 1),
 datetime.date(2026, 2, 28),
 datetime.date(2026, 3, 1),
 datetime.date(2026, 3, 31),
 datetime.date(2026, 4, 1),
 datetime.date(2026, 4, 30),
 datetime.date(2026, 5, 1),
 datetime.date(2026, 5, 31),
 datetime.date(2026, 6, 1),
 datetime.date(2026, 6, 30),
 datetime.date(2026, 7, 1)]

In [31]:
creditor_payments

[8333, 8333, 8333, 8333, 8334, 8334]

In [32]:
date_to_creditor_amount_map = {
    cadence_dates[i]: payment for i, payment in enumerate(creditor_payments)
}

In [33]:
date_to_creditor_amount_map

{datetime.date(2026, 1, 31): 8333,
 datetime.date(2026, 2, 28): 8333,
 datetime.date(2026, 3, 31): 8333,
 datetime.date(2026, 4, 30): 8333,
 datetime.date(2026, 5, 31): 8334,
 datetime.date(2026, 6, 30): 8334}

In [34]:
from feasibility.engine import ScheduleRow

In [35]:
current_escor_balance = client.current_balance_cents
program_fee = round_half_up(offer.original_balance_cents * rules.program_fee_pct)

In [36]:
current_escor_balance

0

In [37]:
rows = []
for date in movement_days:
    if date in cadence_dates:
        amount_to_creditor = date_to_creditor_amount_map[date]
        bank_fee = 0
        if amount_to_creditor > 0:
            bank_fee = rules.bank_fee_cents

        remaining_amount = current_escor_balance - (amount_to_creditor + bank_fee)

        p_fee = min(remaining_amount, program_fee)
        program_fee -= p_fee

        current_escor_balance -= amount_to_creditor + bank_fee + p_fee

        print(
            f"Debited: Creditor {amount_to_creditor} Bank fee: {bank_fee} program_fee: {p_fee}"
        )
        rows.append(
            ScheduleRow(
                date=date,
                creditor_payment_cents=amount_to_creditor,
                program_fee_cents=p_fee,
                bank_fee_cents=bank_fee,
                balance_cents=current_escor_balance,
            )
        )
    elif date in date_to_draft_map:
        draft = date_to_draft_map[date]
        current_escor_balance += draft.amount_cents
        print(f"Credited {draft.amount_cents}. Total: {current_escor_balance}")

Credited 20000. Total: 20000
Debited: Creditor 8333 Bank fee: 1000 program_fee: 10667
Credited 20000. Total: 20000
Debited: Creditor 8333 Bank fee: 1000 program_fee: 10667
Credited 20000. Total: 20000
Debited: Creditor 8333 Bank fee: 1000 program_fee: 8666
Credited 20000. Total: 22001
Debited: Creditor 8333 Bank fee: 1000 program_fee: 0
Credited 20000. Total: 32668
Debited: Creditor 8334 Bank fee: 1000 program_fee: 0
Credited 20000. Total: 43334
Debited: Creditor 8334 Bank fee: 1000 program_fee: 0
Credited 20000. Total: 54000


In [38]:
rows

[ScheduleRow(date=datetime.date(2026, 1, 31), creditor_payment_cents=8333, program_fee_cents=10667, bank_fee_cents=1000, balance_cents=0),
 ScheduleRow(date=datetime.date(2026, 2, 28), creditor_payment_cents=8333, program_fee_cents=10667, bank_fee_cents=1000, balance_cents=0),
 ScheduleRow(date=datetime.date(2026, 3, 31), creditor_payment_cents=8333, program_fee_cents=8666, bank_fee_cents=1000, balance_cents=2001),
 ScheduleRow(date=datetime.date(2026, 4, 30), creditor_payment_cents=8333, program_fee_cents=0, bank_fee_cents=1000, balance_cents=12668),
 ScheduleRow(date=datetime.date(2026, 5, 31), creditor_payment_cents=8334, program_fee_cents=0, bank_fee_cents=1000, balance_cents=23334),
 ScheduleRow(date=datetime.date(2026, 6, 30), creditor_payment_cents=8334, program_fee_cents=0, bank_fee_cents=1000, balance_cents=34000)]

In [39]:
result = Result(feasible=True, schedule=rows, pay_shape_used="even")

In [40]:
def test_case1_feasible_even(r: Result):
    assert r.feasible is True
    assert r.pay_shape_used == "even"
    assert r.schedule is not None
    # balance must never go negative
    assert all(row.balance_cents >= 0 for row in r.schedule)

In [41]:
test_case1_feasible_even(result)

In [42]:
from feasibility.algo import evaluate_offer_pipeline
from feasibility.models import load_case

client, offer, rules = load_case(f"cases/case1_feasible_even")
result = evaluate_offer_pipeline(client, offer, rules)

Creditor payments: [8333, 8333, 8333, 8333, 8334, 8334]
Movement days: [datetime.date(2026, 1, 1), datetime.date(2026, 1, 31), datetime.date(2026, 2, 1), datetime.date(2026, 2, 28), datetime.date(2026, 3, 1), datetime.date(2026, 3, 31), datetime.date(2026, 4, 1), datetime.date(2026, 4, 30), datetime.date(2026, 5, 1), datetime.date(2026, 5, 31), datetime.date(2026, 6, 1), datetime.date(2026, 6, 30), datetime.date(2026, 7, 1)]
Credited 20000. Total: 20000
Debited: Creditor 8333 Bank fee: 1000 p_fee: 10667
Credited 20000. Total: 20000
Debited: Creditor 8333 Bank fee: 1000 p_fee: 10667
Credited 20000. Total: 20000
Debited: Creditor 8333 Bank fee: 1000 p_fee: 8666
Credited 20000. Total: 22001
Debited: Creditor 8333 Bank fee: 1000 p_fee: 0
Credited 20000. Total: 32668
Debited: Creditor 8334 Bank fee: 1000 p_fee: 0
Credited 20000. Total: 43334
Debited: Creditor 8334 Bank fee: 1000 p_fee: 0
Credited 20000. Total: 54000


In [43]:
result.schedule

[ScheduleRow(date=datetime.date(2026, 1, 31), creditor_payment_cents=8333, program_fee_cents=10667, bank_fee_cents=1000, balance_cents=0),
 ScheduleRow(date=datetime.date(2026, 2, 28), creditor_payment_cents=8333, program_fee_cents=10667, bank_fee_cents=1000, balance_cents=0),
 ScheduleRow(date=datetime.date(2026, 3, 31), creditor_payment_cents=8333, program_fee_cents=8666, bank_fee_cents=1000, balance_cents=2001),
 ScheduleRow(date=datetime.date(2026, 4, 30), creditor_payment_cents=8333, program_fee_cents=0, bank_fee_cents=1000, balance_cents=12668),
 ScheduleRow(date=datetime.date(2026, 5, 31), creditor_payment_cents=8334, program_fee_cents=0, bank_fee_cents=1000, balance_cents=23334),
 ScheduleRow(date=datetime.date(2026, 6, 30), creditor_payment_cents=8334, program_fee_cents=0, bank_fee_cents=1000, balance_cents=34000)]

In [1]:
from feasibility.algo import *
from feasibility.models import load_case

In [2]:
def get_cadence_dates(offer: Offer, k_max: int):
    cadence_dates = []
    first_payment_date = offer.first_payment_date
    iter_date = first_payment_date
    i = 0

    is_last_day_ceiling = (
        get_last_day_in_month(first_payment_date.year, first_payment_date.month)
        == first_payment_date.day
    )

    anchor_day = first_payment_date.day

    while i < k_max:
        i += 1
        cadence_dates.append(iter_date)
        iter_date = next_month(iter_date, is_last_day_ceiling, anchor_day)

    return cadence_dates

In [3]:
def calculate_even_pay_payments(total_offer: int, k: int):
    base_amount = total_offer // k
    remaining_cents = total_offer % k

    creditor_payments = [base_amount] * k

    for i in range(remaining_cents):
        creditor_payments[k - 1 - i] += 1

    print(f"Creditor payments: {creditor_payments}")

    return creditor_payments

In [ ]:
client, offer, rules = load_case("cases/case3_balloon")

In [ ]:
total_offer = round_half_up(offer.current_balance_cents * offer.settlement_pct)
program_fee = round_half_up(offer.original_balance_cents * rules.program_fee_pct)

k_max = min(rules.max_terms, rules.max_payments)
cadence_dates = get_cadence_dates(offer, k_max)

date_to_draft_map: dict[datetime.date, list[LedgerEntry]] = {}

for draft in client.ledger:
    date_to_draft_map.setdefault(draft.date, [])
    date_to_draft_map[draft.date].append(draft)

movement_days = cadence_dates + list(date_to_draft_map.keys())
movement_days = sorted(movement_days)

print(f"Movement days: {movement_days}")

if rules.even_pays:
    creditor_payments = calculate_even_pay_payments(total_offer, k_max)
    pay_shape_used = "even"
elif rules.is_ballooning_allowed:
    pay_shape_used = "balloon"
    token_used = rules.max_token_pays - 1
    creditor_payments = [rules.min_payment_cents] * token_used

    remaining_k = k_max - token_used
    amount_rem_to_creditor = total_offer - sum(creditor_payments)

    remaining_payments = calculate_even_pay_payments(
        amount_rem_to_creditor, remaining_k
    )
    creditor_payments += remaining_payments
else:
    pay_shape_used = "staircase"
    if rules.max_segments >= 2:
        creditor_payments = [rules.min_payment_cents] * rules.max_token_pays

        remaining_k = k_max - rules.max_token_pays
        amount_rem_to_creditor = total_offer - sum(creditor_payments)

        remaining_payments = calculate_even_pay_payments(
            amount_rem_to_creditor, remaining_k
        )
        creditor_payments += remaining_payments

date_to_creditor_amount_map = {
    cadence_dates[i]: payment for i, payment in enumerate(creditor_payments)
}

current_escor_balance = client.current_balance_cents
program_fee_remaining = program_fee

rows = []
for date in movement_days:
    if date in date_to_creditor_amount_map:
        amount_to_creditor = date_to_creditor_amount_map[date]
        bank_fee = 0
        if amount_to_creditor > 0:
            bank_fee = rules.bank_fee_cents

        remaining_amount = current_escor_balance - (amount_to_creditor + bank_fee)

        p_fee = min(remaining_amount, program_fee_remaining)
        program_fee_remaining -= p_fee

        current_escor_balance -= amount_to_creditor + bank_fee + p_fee

        print(
            f"Debited: Creditor {amount_to_creditor} Bank fee: {bank_fee} p_fee: {p_fee}"
        )
        rows.append(
            ScheduleRow(
                date=date,
                creditor_payment_cents=amount_to_creditor,
                program_fee_cents=p_fee,
                bank_fee_cents=bank_fee,
                balance_cents=current_escor_balance,
            )
        )
    elif date in date_to_draft_map:
        drafts = date_to_draft_map[date]
        drafts = sorted(
            drafts,
            key=lambda x: int(x.type == "credit"),
            reverse=True,
        )

        for draft in drafts:
            if draft.type == "credit":
                current_escor_balance += draft.amount_cents
                print(f"Credited {draft.amount_cents}. Total: {current_escor_balance}")
            elif draft.type == "debit":
                current_escor_balance -= draft.amount_cents
                print(f"Debited {draft.amount_cents}. Total: {current_escor_balance}")

result = Result(feasible=True, schedule=rows, pay_shape_used=pay_shape_used)

Movement days: [datetime.date(2026, 1, 1), datetime.date(2026, 1, 31), datetime.date(2026, 2, 1), datetime.date(2026, 2, 28), datetime.date(2026, 3, 1), datetime.date(2026, 3, 31), datetime.date(2026, 4, 1), datetime.date(2026, 4, 30), datetime.date(2026, 5, 1), datetime.date(2026, 5, 31), datetime.date(2026, 6, 1), datetime.date(2026, 6, 30), datetime.date(2026, 7, 1)]
Creditor payments: [17500]
Credited 10000. Total: 10000
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 17500
Debited 15000. Total: 2500
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 10000
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 17500
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 25000
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 32500
Debited: Creditor 17500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 25000


In [5]:
result.pay_shape_used

'balloon'

In [6]:
result.schedule

[ScheduleRow(date=datetime.date(2026, 1, 31), creditor_payment_cents=2500, program_fee_cents=0, bank_fee_cents=0, balance_cents=7500),
 ScheduleRow(date=datetime.date(2026, 2, 28), creditor_payment_cents=2500, program_fee_cents=0, bank_fee_cents=0, balance_cents=0),
 ScheduleRow(date=datetime.date(2026, 3, 31), creditor_payment_cents=2500, program_fee_cents=0, bank_fee_cents=0, balance_cents=7500),
 ScheduleRow(date=datetime.date(2026, 4, 30), creditor_payment_cents=2500, program_fee_cents=0, bank_fee_cents=0, balance_cents=15000),
 ScheduleRow(date=datetime.date(2026, 5, 31), creditor_payment_cents=2500, program_fee_cents=0, bank_fee_cents=0, balance_cents=22500),
 ScheduleRow(date=datetime.date(2026, 6, 30), creditor_payment_cents=17500, program_fee_cents=0, bank_fee_cents=0, balance_cents=15000)]

In [7]:
print(result.pay_shape_used)

None


In [ ]:
sorted(
    date_to_draft_map[datetime.date(2026, 2, 1)],
    key=lambda x: int(x.type == "credit"),
    reverse=True,
)

[LedgerEntry(date=datetime.date(2026, 2, 1), amount_cents=10000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 2, 1), amount_cents=15000, type='debit')]

In [13]:
True < False

False

In [ ]:
from feasibility.algo import *
from feasibility.engine import *
from feasibility.models import *

client, offer, rules = load_case("cases/case3_balloon")

In [ ]:
total_offer = round_half_up(offer.current_balance_cents * offer.settlement_pct)
program_fee = round_half_up(offer.original_balance_cents * rules.program_fee_pct)

k_max = min(rules.max_terms, rules.max_payments)
cadence_dates = get_cadence_dates(offer, k_max)

date_to_draft_map: dict[datetime.date, list[LedgerEntry]] = {}

for draft in client.ledger:
    date_to_draft_map.setdefault(draft.date, [])
    date_to_draft_map[draft.date].append(draft)

movement_days = cadence_dates + list(date_to_draft_map.keys())
movement_days = sorted(movement_days)

print(f"Movement days: {movement_days}")

if rules.even_pays:
    creditor_payments = calculate_even_pay_payments(total_offer, k_max)
    pay_shape_used = "even"
elif rules.is_ballooning_allowed:
    pay_shape_used = "balloon"
    token_used = rules.max_token_pays - 1
    creditor_payments = [rules.min_payment_cents] * token_used

    remaining_k = k_max - token_used
    amount_rem_to_creditor = total_offer - sum(creditor_payments)

    remaining_payments = calculate_even_pay_payments(
        amount_rem_to_creditor, remaining_k
    )
    creditor_payments += remaining_payments
else:
    pay_shape_used = "staircase"
    if rules.max_segments >= 2:
        creditor_payments = [rules.min_payment_cents] * rules.max_token_pays

        remaining_k = k_max - rules.max_token_pays
        amount_rem_to_creditor = total_offer - sum(creditor_payments)

        remaining_payments = calculate_even_pay_payments(
            amount_rem_to_creditor, remaining_k
        )
        creditor_payments += remaining_payments

date_to_creditor_amount_map = {
    cadence_dates[i]: payment for i, payment in enumerate(creditor_payments)
}

current_escor_balance = client.current_balance_cents
program_fee_remaining = program_fee

rows = []
for date in movement_days:
    if date in date_to_creditor_amount_map:
        amount_to_creditor = date_to_creditor_amount_map[date]
        bank_fee = 0
        if amount_to_creditor > 0:
            bank_fee = rules.bank_fee_cents

        remaining_amount = current_escor_balance - (amount_to_creditor + bank_fee)

        p_fee = min(remaining_amount, program_fee_remaining)
        program_fee_remaining -= p_fee

        current_escor_balance -= amount_to_creditor + bank_fee + p_fee

        print(
            f"Debited: Creditor {amount_to_creditor} Bank fee: {bank_fee} p_fee: {p_fee}"
        )
        rows.append(
            ScheduleRow(
                date=date,
                creditor_payment_cents=amount_to_creditor,
                program_fee_cents=p_fee,
                bank_fee_cents=bank_fee,
                balance_cents=current_escor_balance,
            )
        )
    elif date in date_to_draft_map:
        drafts = date_to_draft_map[date]
        drafts = sorted(
            drafts,
            key=lambda x: int(x.type == "credit"),
            reverse=True,
        )

        for draft in drafts:
            if draft.type == "credit":
                current_escor_balance += draft.amount_cents
                print(f"Credited {draft.amount_cents}. Total: {current_escor_balance}")
            elif draft.type == "debit":
                current_escor_balance -= draft.amount_cents
                print(f"Debited {draft.amount_cents}. Total: {current_escor_balance}")

result = Result(feasible=True, schedule=rows, pay_shape_used=pay_shape_used)

Movement days: [datetime.date(2026, 1, 1), datetime.date(2026, 1, 31), datetime.date(2026, 2, 1), datetime.date(2026, 2, 28), datetime.date(2026, 3, 1), datetime.date(2026, 3, 31), datetime.date(2026, 4, 1), datetime.date(2026, 4, 30), datetime.date(2026, 5, 1), datetime.date(2026, 5, 31), datetime.date(2026, 6, 1), datetime.date(2026, 6, 30), datetime.date(2026, 7, 1)]
Creditor payments: [17500]
Credited 10000. Total: 10000
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 17500
Debited 15000. Total: 2500
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 10000
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 17500
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 25000
Debited: Creditor 2500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 32500
Debited: Creditor 17500 Bank fee: 0 p_fee: 0
Credited 10000. Total: 25000


In [24]:
from feasibility.models import load_case
import copy

client, offer, rules = load_case(f"cases/case1_feasible_even")

In [11]:
client_2 = copy.deepcopy(client)

In [12]:
client_2.ledger

[LedgerEntry(date=datetime.date(2026, 1, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 2, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 3, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 4, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 5, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 6, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 7, 1), amount_cents=20000, type='credit')]

In [13]:
client_2.ledger = []

In [14]:
client_2.ledger

[]

In [15]:
client.ledger

[LedgerEntry(date=datetime.date(2026, 1, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 2, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 3, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 4, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 5, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 6, 1), amount_cents=20000, type='credit'),
 LedgerEntry(date=datetime.date(2026, 7, 1), amount_cents=20000, type='credit')]

In [21]:
client, offer, rules = load_case(f"cases/case4_tiers")

In [22]:
rules.min_payment_tiers

[(7, 5000), (3, 6000), (3, 5505), (8, 1000)]

In [23]:
sorted(rules.min_payment_tiers, key=lambda x: (x[0], x[1]))

[(3, 5505), (3, 6000), (7, 5000), (8, 1000)]

In [ ]:
from feasibility.engine import evaluate_offer

result = evaluate_offer(client, offer, rules)

In [33]:
schedule = list(sorted(result.schedule, key=lambda x: x.date))
if len(schedule) != len(set([s.date for s in schedule])):
    raise

In [40]:
set([s.date for s in schedule])

{datetime.date(2026, 1, 31),
 datetime.date(2026, 2, 28),
 datetime.date(2026, 3, 31),
 datetime.date(2026, 4, 30),
 datetime.date(2026, 5, 31),
 datetime.date(2026, 6, 30)}

In [1]:
from feasibility.models import load_case, CreditorRules

client, offer, rules = load_case(f"cases/case4_tiers")

In [4]:
def floor_vector(k: int, rules: CreditorRules) -> list[int]:
    minimum_payment_tiers = sorted(rules.min_payment_tiers, key=lambda x: (x[0], x[1]))

    floors = []
    remaining_token_pays = rules.max_token_pays
    active_tier_index = -1
    prev = 0

    for i in range(k):
        while len(minimum_payment_tiers) > (
            active_tier_index + 1
        ) and minimum_payment_tiers[active_tier_index + 1][0] <= (i + 1):
            active_tier_index += 1

        floor = max(rules.min_payment_cents, prev)

        if active_tier_index != -1:
            floor = max(floor, minimum_payment_tiers[active_tier_index][1])

        if floor == rules.min_payment_cents:
            if remaining_token_pays == 0:
                floor = floor + 1
            else:
                remaining_token_pays -= 1

        floors.append(floor)
        prev = floor

    return floors

In [4]:
floors = floor_vector(4, rules)
floors

[2500, 2500, 2500, 2500]

In [ ]:
[23, 45, 66, 144, 700]

In [ ]:
from itertools import combinations


def block_runs(k: int, max_segments: int):
    runs = []
    for num_blocks in range(1, min(k, max_segments) + 1):
        num_cuts = num_blocks - 1
        possibe_cuts = list(combinations(range(1, k), num_cuts))
        # For max_segements = 12, k = 5, num_blocks = 3
        # num_cuts = 2
        # [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
        # (2, 4): This is one combination to cut an array of len 5 at index 2 and 4 to produce 3 blocks
        # blocks = array of block len
        # blocks = [1, 4]: block of len 1 followed by block of len 4
        # blocks = [1, 2, 1, 1]: block of len 1, block of len 2, block of len 1, block of len 1
        # runs for num_cuts = 2:  [[2, 3], [3, 2], [4, 1]] --> Basically all possible combination

        for cut_positions in possibe_cuts:
            blocks = []

            prev_cut_position = 0
            for position_to_cut in cut_positions:
                blocks.append(position_to_cut - prev_cut_position)
                prev_cut_position = position_to_cut

            blocks.append(k - prev_cut_position)

            if k > 1 and blocks[-1] < 2:
                continue

            runs.append(blocks)

    return runs

In [6]:
runs = block_runs(5, 12)
runs

[[5],
 [1, 4],
 [2, 3],
 [3, 2],
 [4, 1],
 [1, 1, 3],
 [1, 2, 2],
 [1, 3, 1],
 [2, 1, 2],
 [2, 2, 1],
 [3, 1, 1],
 [1, 1, 1, 2],
 [1, 1, 2, 1],
 [1, 2, 1, 1],
 [2, 1, 1, 1],
 [1, 1, 1, 1, 1]]

In [7]:
# [23, 45, 66, 144, 700]

In [6]:
from feasibility.utils import round_half_up


def build_from_run(run: list[int], floors: list[int], total_offer: int) -> list[int]:
    num_blocks = len(run)

    block_ends_index = []
    len_till_now = 0
    for block_len in run:
        len_till_now += block_len
        block_ends_index.append(len_till_now - 1)

    unique_values_per_block = []
    prev_block_value = 0
    for i in range(num_blocks - 1):
        block_value = max(prev_block_value + 1, floors[block_ends_index[i]])
        unique_values_per_block.append(block_value)
        prev_block_value = block_value

    leftover_cents = total_offer - sum(
        run[i] * unique_values_per_block[i] for i in range(num_blocks - 1)
    )
    last_block_len = run[-1]

    if leftover_cents % last_block_len:
        is_fixed = False

        for i in range(num_blocks - 2, -1, -1):
            ceiling = (
                unique_values_per_block[i + 1] - 1
                if (i + 1) < len(unique_values_per_block)
                else None
            )

            for delta_cents in range(1, last_block_len):
                modified_value = unique_values_per_block[i] + delta_cents

                if ceiling is not None and modified_value > ceiling:
                    break

                if (leftover_cents - delta_cents * run[i]) % last_block_len == 0:
                    unique_values_per_block[i] = modified_value
                    is_fixed = True
                    leftover_cents -= delta_cents * run[i]
                    break

            if is_fixed:
                break

        if not is_fixed:
            return []

    last_block_value = leftover_cents // last_block_len

    if (
        len(unique_values_per_block) >= 1
        and last_block_value <= unique_values_per_block[-1]
    ):
        return []

    if last_block_value < floors[block_ends_index[-1]]:
        return []

    unique_values_per_block.append(last_block_value)

    creditor_payments = []

    for i in range(num_blocks):
        creditor_payments += [unique_values_per_block[i]] * run[i]

    return creditor_payments

In [2]:
k = 3
total_offer = round_half_up(offer.current_balance_cents * offer.settlement_pct)
# runs = block_runs(k, rules.max_segments)
# floors = floor_vector(k, rules)

# print(total_offer, floors, runs)
# build_from_run(runs[1], floors, total_offer)

NameError: name 'round_half_up' is not defined

In [17]:
def staircase_payments(k: int, total_offer: int, rules: CreditorRules):
    floors = floor_vector(k, rules)

    if sum(floors) > total_offer:
        return []
    elif sum(floors) == total_offer:
        return floors

    runs = block_runs(k, rules.max_segments)
    best = None

    for run in runs:
        payment_vector = build_from_run(run, floors, total_offer)

        if not payment_vector:
            continue

        # if not is_vali

        best = min(payment_vector, best) if best is not None else payment_vector

    return best

In [18]:
min([3, 4, 55, 234], [3, 4, 32, 123])

[3, 4, 32, 123]

In [19]:
staircase_payments(k, total_offer, rules)

[2500, 28750, 28750]

In [1]:
from feasibility.models import load_case
from feasibility.utils import round_half_up
from feasibility.staircase import staircase_payments

client, offer, rules = load_case(f"cases/case4_tiers")

k = 3
total_offer = round_half_up(offer.current_balance_cents * offer.settlement_pct)

staircase_payments(k, total_offer, rules)

[2500, 28750, 28750]